# STKDE Parameter Sensitivity Experiments

## Space-Time Kernel Density Estimation for Chicago Crime Hotspot Analysis

**Thesis Research Notebook**  
This notebook systematically experiments with STKDE parameters to understand how kernel size, bandwidth, and grid resolution affect hotspot detection quality, stability, and interpretability for the adaptive space-time cube prototype.

### Research Questions
1. How does **spatial bandwidth** affect hotspot size, count, and spatial distribution?
2. How does **temporal bandwidth** influence hotspot temporal precision?
3. What is the effect of **grid cell resolution** on computational cost vs. spatial fidelity?
4. Which **kernel function** (Gaussian, Epanechnikov, Uniform, Quartic) produces the most interpretable results?
5. What is the **stability** of top-K hotspots under parameter perturbation?

## 0. Environment Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from matplotlib.patches import FancyBboxPatch
import seaborn as sns
from pathlib import Path
from datetime import datetime, timedelta
import time
import json
import warnings
from itertools import product
from collections import defaultdict

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', context='notebook', palette='muted')
plt.rcParams['figure.dpi'] = 150
plt.rcParams['savefig.dpi'] = 300
plt.rcParams['font.size'] = 10
plt.rcParams['axes.titlesize'] = 12
plt.rcParams['axes.labelsize'] = 10

print('Environment ready.')

## 1. Data Loading & Preprocessing

Load the Chicago crimes dataset (2015-2025, ~2.7M records after cleaning).

In [ ]:
# Configuration
DATA_PATH = Path('data/Crimes_-_2001_to_Present_20260114.csv')
OUTPUT_DIR = Path('stkde_experiment_output')
OUTPUT_DIR.mkdir(exist_ok=True)

# Chicago bounds
CITY_BOUNDS = {
    'lat_min': 41.644, 'lat_max': 42.023,
    'lon_min': -87.940, 'lon_max': -87.524,
}

# Study period
STUDY_START = '2020-01-01'
STUDY_END = '2020-12-31'

print(f'Data path: {DATA_PATH.resolve()}')
print(f'Study period: {STUDY_START} to {STUDY_END}')

In [ ]:
DTYPES = {
    'ID': 'int64',
    'Date': 'string',
    'Primary Type': 'string',
    'District': 'Int64',
    'Latitude': 'float64',
    'Longitude': 'float64',
}

def load_and_clean_crimes(path, start_date, end_date, max_rows=None):
    """Load and clean crime data for STKDE experiments."""
    print(f'Loading CSV...')
    df = pd.read_csv(path, dtype=DTYPES, usecols=list(DTYPES.keys()),
                     nrows=max_rows, na_values={'': None})
    
    # Parse dates
    df['Date'] = pd.to_datetime(df['Date'], errors='coerce')
    
    # Filter invalid coordinates
    df = df.dropna(subset=['Latitude', 'Longitude'])
    df = df[(df['Latitude'] != 0) & (df['Longitude'] != 0)]
    
    # Filter to Chicago bounds
    df = df[
        (df['Latitude'].between(CITY_BOUNDS['lat_min'], CITY_BOUNDS['lat_max'])) &
        (df['Longitude'].between(CITY_BOUNDS['lon_min'], CITY_BOUNDS['lon_max']))
    ]
    
    # Filter to study period
    df = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)]
    
    # Add epoch seconds for temporal computation
    df['epoch_sec'] = df['Date'].astype('int64') // 10**9
    
    print(f'Loaded {len(df):,} valid crime records')
    print(f'Date range: {df["Date"].min()} to {df["Date"].max()}')
    print(f'Crime types: {df["Primary Type"].nunique()}')
    return df

# Load one year of data for experiments (adjust for more/less)
crimes_df = load_and_clean_crimes(DATA_PATH, STUDY_START, STUDY_END, max_rows=None)
crimes_df[['Latitude', 'Longitude', 'Date', 'Primary Type']].head(10)

In [ ]:
# Quick EDA for the experiment subset
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# Spatial distribution
axes[0].scatter(crimes_df['Longitude'].sample(min(50000, len(crimes_df))),
                crimes_df['Latitude'].sample(min(50000, len(crimes_df))),
                s=1, alpha=0.3, c='steelblue')
axes[0].set_title('Spatial Distribution (sample)')
axes[0].set_xlabel('Longitude'); axes[0].set_ylabel('Latitude')

# Temporal distribution
crimes_df.set_index('Date').resample('W').size().plot(ax=axes[1], color='steelblue')
axes[1].set_title('Weekly Crime Counts')
axes[1].set_xlabel('Date'); axes[1].set_ylabel('Count')

# Top crime types
top_types = crimes_df['Primary Type'].value_counts().head(10)
axes[2].barh(top_types.index, top_types.values, color='steelblue')
axes[2].set_title('Top 10 Crime Types')
axes[2].set_xlabel('Count'); axes[2].invert_yaxis()

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'eda_overview.png', bbox_inches='tight')
plt.show()
print(f'\nDataset: {len(crimes_df):,} records, {crimes_df["Primary Type"].nunique()} crime types')

## 2. Core STKDE Implementation

Pure Python/numpy implementation matching the TypeScript engine in `src/lib/stkde/compute.ts`.  
Key components: spatial grid → support accumulation → kernel smoothing → intensity normalization → hotspot ranking.

In [ ]:
class STKDEConfig:
    """STKDE parameter configuration."""
    def __init__(self, spatial_bw_m=750, temporal_bw_h=24, grid_cell_m=500,
                 top_k=12, min_support=5, time_window_h=24, kernel='gaussian',
                 bbox=None):
        self.spatial_bandwidth_m = spatial_bw_m      # Spatial kernel bandwidth (meters)
        self.temporal_bandwidth_h = temporal_bw_h    # Temporal kernel bandwidth (hours)
        self.grid_cell_m = grid_cell_m               # Grid cell size (meters)
        self.top_k = top_k                           # Number of top hotspots
        self.min_support = min_support               # Minimum events per cell
        self.time_window_h = time_window_h           # Peak window width (hours)
        self.kernel = kernel                         # Kernel function type
        self.bbox = bbox or [CITY_BOUNDS['lon_min'], CITY_BOUNDS['lat_min'],
                              CITY_BOUNDS['lon_max'], CITY_BOUNDS['lat_max']]
    
    def to_dict(self):
        return {k: v for k, v in self.__dict__.items() if not k.startswith('_')}
    
    def __repr__(self):
        return (f'STKDEConfig(sbw={self.spatial_bandwidth_m}m, tbw={self.temporal_bandwidth_h}h, '
                f'grid={self.grid_cell_m}m, k={self.top_k}, min_sup={self.min_support}, '
                f'kernel={self.kernel})')

# Constants
METERS_PER_LAT_DEGREE = 111_320

def meters_to_lon_degrees(meters, lat):
    """Convert meters to degrees longitude at given latitude."""
    return meters / (METERS_PER_LAT_DEGREE * np.cos(np.radians(lat)))

def meters_to_lat_degrees(meters):
    """Convert meters to degrees latitude."""
    return meters / METERS_PER_LAT_DEGREE

print('STKDE config class and spatial conversion utilities defined.')

In [ ]:
def gaussian_kernel(distance, sigma):
    """Gaussian kernel: K(u) = exp(-0.5 * (u/sigma)^2)"""
    return np.exp(-0.5 * (distance / sigma) ** 2)

def epanechnikov_kernel(distance, bandwidth):
    """Epanechnikov kernel: K(u) = 0.75 * (1 - u^2) for |u| <= 1"""
    u = distance / bandwidth
    result = np.where(np.abs(u) <= 1, 0.75 * (1 - u ** 2), 0)
    return result

def uniform_kernel(distance, bandwidth):
    """Uniform kernel: K(u) = 0.5 for |u| <= 1"""
    u = distance / bandwidth
    return np.where(np.abs(u) <= 1, 0.5, 0)

def quartic_biweight_kernel(distance, bandwidth):
    """Quartic (biweight) kernel: K(u) = (15/16) * (1 - u^2)^2 for |u| <= 1"""
    u = distance / bandwidth
    result = np.where(np.abs(u) <= 1, (15/16) * (1 - u ** 2) ** 2, 0)
    return result

def triangular_kernel(distance, bandwidth):
    """Triangular kernel: K(u) = 1 - |u| for |u| <= 1"""
    u = distance / bandwidth
    return np.where(np.abs(u) <= 1, 1 - np.abs(u), 0)

KERNEL_FUNCTIONS = {
    'gaussian': gaussian_kernel,
    'epanechnikov': epanechnikov_kernel,
    'uniform': uniform_kernel,
    'quartic': quartic_biweight_kernel,
    'triangular': triangular_kernel,
}

# Visualize kernels
x = np.linspace(-3, 3, 300)
fig, ax = plt.subplots(figsize=(10, 5))
for name, kernel_fn in KERNEL_FUNCTIONS.items():
    if name == 'gaussian':
        y = kernel_fn(np.abs(x), 1.0)
    else:
        y = kernel_fn(np.abs(x), 1.0)
    ax.plot(x, y, label=name, linewidth=2)
ax.set_title('Kernel Function Comparison')
ax.set_xlabel('Distance (normalized)'); ax.set_ylabel('Weight')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'kernel_functions.png', bbox_inches='tight')
plt.show()

In [ ]:
class STKDEEngine:
    """Space-Time Kernel Density Estimation engine.
    
    Implements the same algorithm as src/lib/stkde/compute.ts.
    """
    
    def __init__(self, config: STKDEConfig):
        self.config = config
        self._grid = None
        self._support = None
        self._intensity = None
        self._hotspots = None
        self._heatmap_cells = None
    
    def build_grid(self):
        """Build spatial grid from config."""
        cfg = self.config
        min_lon, min_lat, max_lon, max_lat = cfg.bbox
        mean_lat = (min_lat + max_lat) / 2
        
        lat_cell_deg = meters_to_lat_degrees(cfg.grid_cell_m)
        lon_cell_deg = meters_to_lon_degrees(cfg.grid_cell_m, mean_lat)
        
        lat_span = max(1e-6, max_lat - min_lat)
        lon_span = max(1e-6, max_lon - min_lon)
        
        rows = max(1, int(np.ceil(lat_span / lat_cell_deg)))
        cols = max(1, int(np.ceil(lon_span / lon_cell_deg)))
        
        self._grid = {
            'min_lon': min_lon, 'min_lat': min_lat,
            'max_lon': max_lon, 'max_lat': max_lat,
            'rows': rows, 'cols': cols,
            'lat_cell_deg': lat_span / rows,
            'lon_cell_deg': lon_span / cols,
            'mean_lat': mean_lat,
        }
        return self._grid
    
    def _cell_index(self, lon, lat):
        """Map (lon, lat) to grid cell index."""
        g = self._grid
        col = int(np.clip((lon - g['min_lon']) / g['lon_cell_deg'], 0, g['cols'] - 1))
        row = int(np.clip((lat - g['min_lat']) / g['lat_cell_deg'], 0, g['rows'] - 1))
        return row * g['cols'] + col
    
    def _cell_centroid(self, idx):
        """Get (lon, lat) centroid of grid cell."""
        g = self._grid
        row = idx // g['cols']
        col = idx % g['cols']
        lon = g['min_lon'] + (col + 0.5) * g['lon_cell_deg']
        lat = g['min_lat'] + (row + 0.5) * g['lat_cell_deg']
        return lon, lat
    
    def _cell_distance_meters(self, idx1, idx2):
        """Compute distance in meters between two cell centroids."""
        lon1, lat1 = self._cell_centroid(idx1)
        lon2, lat2 = self._cell_centroid(idx2)
        # Approximate using equirectangular projection
        dlat = (lat2 - lat1) * METERS_PER_LAT_DEGREE
        dlon = (lon2 - lon1) * METERS_PER_LAT_DEGREE * np.cos(np.radians((lat1 + lat2) / 2))
        return np.sqrt(dlat ** 2 + dlon ** 2)
    
    def compute_support(self, lons, lats, timestamps):
        """Bin events into grid cells and accumulate support counts."""
        if self._grid is None:
            self.build_grid()
        
        n_cells = self._grid['rows'] * self._grid['cols']
        support = np.zeros(n_cells, dtype=np.float64)
        cell_timestamps = [[] for _ in range(n_cells)]
        
        for lon, lat, ts in zip(lons, lats, timestamps):
            idx = self._cell_index(lon, lat)
            support[idx] += 1
            cell_timestamps[idx].append(ts)
        
        self._support = support
        self._cell_timestamps = cell_timestamps
        return support, cell_timestamps
    
    def smooth_intensity(self, kernel_fn=None):
        """Apply spatial kernel smoothing to support counts."""
        if self._support is None:
            raise ValueError('Must compute_support() first')
        
        g = self._grid
        cfg = self.config
        n_cells = g['rows'] * g['cols']
        
        # Kernel radius in cells
        bandwidth_cells = max(1, int(np.ceil(cfg.spatial_bandwidth_m / cfg.grid_cell_m)))
        sigma_cells = max(0.5, bandwidth_cells / 2)
        kernel_radius = max(1, int(np.ceil(3 * sigma_cells)))
        
        if kernel_fn is None:
            kernel_fn = KERNEL_FUNCTIONS.get(cfg.kernel, gaussian_kernel)
        
        intensity = np.zeros(n_cells, dtype=np.float64)
        
        # Precompute cell centroids
        cell_lons = np.zeros(n_cells)
        cell_lats = np.zeros(n_cells)
        for i in range(n_cells):
            cell_lons[i], cell_lats[i] = self._cell_centroid(i)
        
        for row in range(g['rows']):
            for col in range(g['cols']):
                center_idx = row * g['cols'] + col
                r_start = max(0, row - kernel_radius)
                r_end = min(g['rows'], row + kernel_radius + 1)
                c_start = max(0, col - kernel_radius)
                c_end = min(g['cols'], col + kernel_radius + 1)
                
                for r in range(r_start, r_end):
                    for c in range(c_start, c_end):
                        neighbor_idx = r * g['cols'] + c
                        count = self._support[neighbor_idx]
                        if count <= 0:
                            continue
                        # Cell-grid distance
                        dr = r - row
                        dc = c - col
                        distance_cells = np.sqrt(dr ** 2 + dc ** 2)
                        
                        if cfg.kernel == 'gaussian':
                            weight = gaussian_kernel(distance_cells, sigma_cells)
                        else:
                            weight = kernel_fn(distance_cells, bandwidth_cells)
                        
                        intensity[center_idx] += count * weight
        
        self._intensity = intensity
        self._max_intensity = intensity.max() if intensity.max() > 0 else 1.0
        self._bandwidth_cells = bandwidth_cells
        self._sigma_cells = sigma_cells
        return intensity
    
    def extract_hotspots(self):
        """Extract and rank hotspots from intensity surface."""
        if self._intensity is None:
            raise ValueError('Must smooth_intensity() first')
        
        g = self._grid
        cfg = self.config
        n_cells = g['rows'] * g['cols']
        
        candidates = []
        for idx in range(n_cells):
            support_count = int(self._support[idx])
            if support_count < cfg.min_support:
                continue
            
            lon, lat = self._cell_centroid(idx)
            norm_intensity = float(self._intensity[idx] / self._max_intensity)
            
            # Find peak time window
            timestamps = self._cell_timestamps[idx]
            peak_start, peak_end = self._find_peak_window(timestamps)
            
            candidates.append({
                'id': f'hs-{idx}',
                'lon': round(lon, 6),
                'lat': round(lat, 6),
                'intensity': round(norm_intensity, 6),
                'support': support_count,
                'intensity_raw': float(self._intensity[idx]),
                'peak_start_epoch': peak_start,
                'peak_end_epoch': peak_end,
                'radius_m': cfg.spatial_bandwidth_m,
                'grid_row': idx // g['cols'],
                'grid_col': idx % g['cols'],
            })
        
        # Sort by intensity desc, support desc
        candidates.sort(key=lambda x: (x['intensity'], x['support']), reverse=True)
        self._hotspots = candidates[:cfg.top_k]
        
        # Build heatmap cells (all non-zero cells)
        heatmap_cells = []
        for idx in range(n_cells):
            if self._support[idx] > 0 or self._intensity[idx] > 0:
                lon, lat = self._cell_centroid(idx)
                heatmap_cells.append({
                    'lon': round(lon, 6),
                    'lat': round(lat, 6),
                    'intensity': round(float(self._intensity[idx] / self._max_intensity), 6),
                    'support': int(self._support[idx]),
                })
        self._heatmap_cells = heatmap_cells
        
        return self._hotspots
    
    def _find_peak_window(self, timestamps):
        """Find the densest time window for a cell's events."""
        window_sec = self.config.time_window_h * 3600
        if len(timestamps) == 0:
            return 0, window_sec
        
        sorted_ts = sorted(timestamps)
        best_start = sorted_ts[0]
        best_count = 1
        start_idx = 0
        
        for end_idx in range(len(sorted_ts)):
            while sorted_ts[end_idx] - sorted_ts[start_idx] > window_sec:
                start_idx += 1
            count = end_idx - start_idx + 1
            if count > best_count:
                best_count = count
                best_start = sorted_ts[start_idx]
        
        return best_start, best_start + window_sec
    
    def run(self, lons, lats, timestamps):
        """Run complete STKDE pipeline."""
        t0 = time.time()
        self.compute_support(lons, lats, timestamps)
        self.smooth_intensity()
        hotspots = self.extract_hotspots()
        elapsed_ms = (time.time() - t0) * 1000
        
        stats = {
            'n_events': len(lons),
            'n_cells': self._grid['rows'] * self._grid['cols'],
            'grid_rows': self._grid['rows'],
            'grid_cols': self._grid['cols'],
            'n_active_cells': int((self._support > 0).sum()),
            'n_hotspot_candidates': len(self._heatmap_cells),
            'n_hotspots': len(hotspots),
            'max_intensity': float(self._max_intensity),
            'max_support': int(self._support.max()),
            'bandwidth_cells': self._bandwidth_cells,
            'sigma_cells': self._sigma_cells,
            'compute_ms': round(elapsed_ms, 1),
        }
        return hotspots, stats

print('STKDE Engine implemented (matching TypeScript compute.ts logic).')

In [ ]:
# Quick smoke test with default parameters
test_config = STKDEConfig(
    spatial_bw_m=750, temporal_bw_h=24, grid_cell_m=500,
    top_k=12, min_support=5, time_window_h=24, kernel='gaussian'
)

# Use a subset for quick testing
test_df = crimes_df.sample(n=50000, random_state=42)

engine = STKDEEngine(test_config)
hotspots, stats = engine.run(
    test_df['Longitude'].values,
    test_df['Latitude'].values,
    test_df['epoch_sec'].values
)

print('=== Smoke Test Results ===')
print(json.dumps(stats, indent=2))
print(f'\nTop 5 Hotspots:')
for i, h in enumerate(hotspots[:5]):
    print(f'  #{i+1}: ({h["lon"]:.4f}, {h["lat"]:.4f}) intensity={h["intensity"]:.4f} support={h["support"]}')

## 3. Visualization Utilities

In [ ]:
def plot_hotspot_map(lons, lats, hotspots, config, ax=None, title=None):
    """Plot crime points and hotspot overlay on a map."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 8))
    
    # Sample points for rendering
    n_sample = min(30000, len(lons))
    idx = np.random.choice(len(lons), n_sample, replace=False)
    ax.scatter(lons[idx], lats[idx], s=1, alpha=0.15, c='#555555', rasterized=True)
    
    # Plot hotspot circles
    for h in hotspots:
        radius_deg = meters_to_lat_degrees(h['radius_m'])
        circle = plt.Circle((h['lon'], h['lat']), radius_deg,
                          facecolor='red', edgecolor='darkred',
                          alpha=0.25, linewidth=1.5)
        ax.add_patch(circle)
        ax.annotate(str(hotspots.index(h) + 1),
                   (h['lon'], h['lat']),
                   fontsize=7, ha='center', va='center',
                   color='white', fontweight='bold',
                   bbox=dict(boxstyle='circle,pad=0.15', facecolor='darkred', alpha=0.9))
    
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_aspect(1 / np.cos(np.radians(np.mean(lats))))
    if title:
        ax.set_title(title, fontweight='bold')
    else:
        ax.set_title(f'STKDE Hotspots (SBW={config.spatial_bandwidth_m}m, '
                    f'Grid={config.grid_cell_m}m, K={config.top_k})', fontweight='bold')
    return ax


def plot_hotspot_comparison(results_dict, lons, lats, n_cols=3, figsize=(18, 5)):
    """Compare hotspot results across multiple configurations."""
    n = len(results_dict)
    n_rows = int(np.ceil(n / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(figsize[0], figsize[1] * n_rows))
    if n == 1:
        axes = [axes]
    axes = np.atleast_1d(axes).flatten()
    
    for ax, (label, (hotspots, config)) in zip(axes, results_dict.items()):
        plot_hotspot_map(lons, lats, hotspots, config, ax=ax, title=label)
    
    for ax in axes[n:]:
        ax.set_visible(False)
    
    plt.tight_layout()
    return fig


def plot_intensity_heatmap(engine, lons, lats, ax=None, title=None):
    """Plot the intensity heatmap surface."""
    if ax is None:
        _, ax = plt.subplots(figsize=(10, 8))
    
    g = engine._grid
    intensity_grid = engine._intensity.reshape(g['rows'], g['cols'])
    
    # Create mesh
    lon_edges = np.linspace(g['min_lon'], g['max_lon'], g['cols'] + 1)
    lat_edges = np.linspace(g['min_lat'], g['max_lat'], g['rows'] + 1)
    
    im = ax.pcolormesh(lon_edges, lat_edges, intensity_grid / engine._max_intensity,
                      cmap='YlOrRd', shading='flat', alpha=0.8)
    plt.colorbar(im, ax=ax, label='Normalized Intensity', shrink=0.8)
    
    ax.set_xlabel('Longitude'); ax.set_ylabel('Latitude')
    ax.set_aspect(1 / np.cos(np.radians(np.mean(lats))))
    if title:
        ax.set_title(title, fontweight='bold')
    return ax

print('Visualization utilities defined.')

---
## 4. Experiment 1: Spatial Bandwidth Sensitivity

**Research Question:** How does spatial bandwidth affect hotspot size, count, and spatial distribution?

**Method:** Fix all parameters except spatial bandwidth, sweep from 200m to 3000m.

In [ ]:
# Experiment 1: Vary spatial bandwidth
SPATIAL_BW_VALUES = [200, 400, 750, 1200, 2000, 3000]

# Use consistent data subset for fair comparison
exp1_df = crimes_df.sample(n=100000, random_state=42)
exp1_lons = exp1_df['Longitude'].values
exp1_lats = exp1_df['Latitude'].values
exp1_ts = exp1_df['epoch_sec'].values

exp1_results = {}
exp1_stats = []

for sbw in SPATIAL_BW_VALUES:
    config = STKDEConfig(
        spatial_bw_m=sbw, temporal_bw_h=24, grid_cell_m=500,
        top_k=15, min_support=5, time_window_h=24, kernel='gaussian'
    )
    engine = STKDEEngine(config)
    hotspots, stats = engine.run(exp1_lons, exp1_lats, exp1_ts)
    stats['spatial_bw_m'] = sbw
    exp1_results[f'SBW={sbw}m'] = (hotspots, config)
    exp1_stats.append(stats)
    print(f'  SBW={sbw}m: {len(hotspots)} hotspots, n_active={stats["n_active_cells"]}, '
          f'bandwidth_cells={stats["bandwidth_cells"]}, compute={stats["compute_ms"]}ms')

print('\nExperiment 1 complete.')

In [ ]:
# Visualize hotspot comparison
fig = plot_hotspot_comparison(exp1_results, exp1_lons, exp1_lats, n_cols=3, figsize=(18, 11))
fig.suptitle('Experiment 1: Spatial Bandwidth Sensitivity', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp1_spatial_bw_hotspots.png', bbox_inches='tight')
plt.show()

In [ ]:
# Quantitative analysis
exp1_df_stats = pd.DataFrame(exp1_stats)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Active cells vs bandwidth
axes[0, 0].plot(exp1_df_stats['spatial_bw_m'], exp1_df_stats['n_active_cells'], 
               'o-', color='steelblue', markersize=8)
axes[0, 0].set_title('Active Cells vs Spatial BW')
axes[0, 0].set_xlabel('Spatial Bandwidth (m)'); axes[0, 0].set_ylabel('Active Cells')

# Hotspots vs bandwidth
axes[0, 1].plot(exp1_df_stats['spatial_bw_m'], exp1_df_stats['n_hotspots'],
               's-', color='coral', markersize=8)
axes[0, 1].set_title('Hotspot Count vs Spatial BW')
axes[0, 1].set_xlabel('Spatial Bandwidth (m)'); axes[0, 1].set_ylabel('Hotspots')

# Max intensity vs bandwidth
axes[0, 2].plot(exp1_df_stats['spatial_bw_m'], exp1_df_stats['max_intensity'],
               '^-', color='green', markersize=8)
axes[0, 2].set_title('Max Intensity vs Spatial BW')
axes[0, 2].set_xlabel('Spatial Bandwidth (m)'); axes[0, 2].set_ylabel('Max Intensity')

# Compute time vs bandwidth
axes[1, 0].plot(exp1_df_stats['spatial_bw_m'], exp1_df_stats['compute_ms'],
               'D-', color='purple', markersize=8)
axes[1, 0].set_title('Compute Time vs Spatial BW')
axes[1, 0].set_xlabel('Spatial Bandwidth (m)'); axes[1, 0].set_ylabel('Compute Time (ms)')

# Bandwidth cells vs spatial bw
axes[1, 1].plot(exp1_df_stats['spatial_bw_m'], exp1_df_stats['bandwidth_cells'],
               'p-', color='brown', markersize=8)
axes[1, 1].set_title('Kernel Radius (cells) vs Spatial BW')
axes[1, 1].set_xlabel('Spatial Bandwidth (m)'); axes[1, 1].set_ylabel('Kernel Radius (cells)')

# Max support vs bandwidth
axes[1, 2].plot(exp1_df_stats['spatial_bw_m'], exp1_df_stats['max_support'],
               'h-', color='teal', markersize=8)
axes[1, 2].set_title('Max Support vs Spatial BW')
axes[1, 2].set_xlabel('Spatial Bandwidth (m)'); axes[1, 2].set_ylabel('Max Support')

plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp1_spatial_bw_metrics.png', bbox_inches='tight')
plt.show()

In [ ]:
# Hotspot stability: Jaccard similarity between adjacent bandwidths
def jaccard_similarity(hotspots_a, hotspots_b, distance_threshold_m=500):
    """Compute Jaccard index based on spatial proximity of hotspot centroids."""
    if not hotspots_a or not hotspots_b:
        return 0.0
    
    matched = 0
    for ha in hotspots_a:
        for hb in hotspots_b:
            dlat = (ha['lat'] - hb['lat']) * METERS_PER_LAT_DEGREE
            dlon = (ha['lon'] - hb['lon']) * METERS_PER_LAT_DEGREE * np.cos(np.radians((ha['lat'] + hb['lat']) / 2))
            dist = np.sqrt(dlat**2 + dlon**2)
            if dist < distance_threshold_m:
                matched += 1
                break
    
    union = len(hotspots_a) + len(hotspots_b) - matched
    return matched / union if union > 0 else 0.0

hotspot_lists = [v[0] for v in exp1_results.values()]
similarities = []
for i in range(len(hotspot_lists) - 1):
    sim = jaccard_similarity(hotspot_lists[i], hotspot_lists[i + 1])
    similarities.append(sim)

fig, ax = plt.subplots(figsize=(8, 4))
labels = [f'{SPATIAL_BW_VALUES[i]}->{SPATIAL_BW_VALUES[i+1]}' for i in range(len(SPATIAL_BW_VALUES)-1)]
ax.bar(labels, similarities, color='steelblue', alpha=0.8)
ax.set_title('Hotspot Stability: Jaccard Similarity Between Adjacent Bandwidths')
ax.set_ylabel('Jaccard Index')
ax.set_ylim(0, 1)
ax.axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='Moderate similarity')
ax.legend()
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp1_stability.png', bbox_inches='tight')
plt.show()

print('Experiment 1 Summary:')
print(f'  Spatial BW range: {SPATIAL_BW_VALUES[0]}-{SPATIAL_BW_VALUES[-1]}m')
print(f'  Hotspot count range: {min(s["n_hotspots"] for s in exp1_stats)}-{max(s["n_hotspots"] for s in exp1_stats)}')
print(f'  Mean Jaccard similarity: {np.mean(similarities):.3f}')

---
## 5. Experiment 2: Temporal Bandwidth Sensitivity

**Research Question:** How does temporal bandwidth influence hotspot detection?

**Method:** Fix spatial parameters, sweep temporal bandwidth from 4h to 168h (1 week).

In [ ]:
TEMPORAL_BW_VALUES = [4, 12, 24, 48, 72, 168]  # hours

exp2_results = {}
exp2_stats = []

for tbw in TEMPORAL_BW_VALUES:
    config = STKDEConfig(
        spatial_bw_m=750, temporal_bw_h=tbw, grid_cell_m=500,
        top_k=15, min_support=5, time_window_h=tbw, kernel='gaussian'
    )
    engine = STKDEEngine(config)
    hotspots, stats = engine.run(exp1_lons, exp1_lats, exp1_ts)
    stats['temporal_bw_h'] = tbw
    exp2_results[f'TBW={tbw}h'] = (hotspots, config)
    exp2_stats.append(stats)
    print(f'  TBW={tbw}h: {len(hotspots)} hotspots, max_intensity={stats["max_intensity"]:.1f}, '
          f'compute={stats["compute_ms"]}ms')

print('\nExperiment 2 complete.')

In [ ]:
# Temporal bandwidth affects peak window detection primarily.
# Here we analyze the hotspot peak time distribution.

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
axes = axes.flatten()

for ax, (tbw, (hotspots, config)) in zip(axes, exp2_results.items()):
    peak_durations_h = [(h['peak_end_epoch'] - h['peak_start_epoch']) / 3600 for h in hotspots]
    intensities = [h['intensity'] for h in hotspots]
    
    ax.scatter(peak_durations_h, intensities, alpha=0.7, s=50,
              c=np.arange(len(hotspots)), cmap='viridis')
    ax.set_title(f'TBW={tbw}h (n={len(hotspots)} hotspots)')
    ax.set_xlabel('Peak Duration (h)'); ax.set_ylabel('Intensity')
    ax.set_xlim(0, max(peak_durations_h) * 1.1 if peak_durations_h else 1)

plt.suptitle('Experiment 2: Temporal Bandwidth Effects on Peak Windows', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp2_temporal_bw.png', bbox_inches='tight')
plt.show()

---
## 6. Experiment 3: Grid Resolution Effects

**Research Question:** How does grid cell size affect spatial fidelity and computational cost?

**Method:** Vary grid cell size from 100m to 2000m, tracking cell count, active cells, and compute time.

In [ ]:
GRID_CELL_VALUES = [100, 250, 500, 750, 1000, 1500, 2000]

exp3_results = {}
exp3_stats = []

for gcm in GRID_CELL_VALUES:
    config = STKDEConfig(
        spatial_bw_m=750, temporal_bw_h=24, grid_cell_m=gcm,
        top_k=15, min_support=5, time_window_h=24, kernel='gaussian'
    )
    engine = STKDEEngine(config)
    hotspots, stats = engine.run(exp1_lons, exp1_lats, exp1_ts)
    stats['grid_cell_m'] = gcm
    exp3_results[f'Grid={gcm}m'] = (hotspots, config)
    exp3_stats.append(stats)
    print(f'  Grid={gcm}m: {stats["grid_rows"]}x{stats["grid_cols"]} '
          f'({stats["n_cells"]:,} cells), {stats["n_hotspots"]} hotspots, '
          f'compute={stats["compute_ms"]}ms')

print('\nExperiment 3 complete.')

In [ ]:
exp3_df = pd.DataFrame(exp3_stats)

fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# Grid cells vs grain
axes[0, 0].plot(exp3_df['grid_cell_m'], exp3_df['n_cells'], 'o-', color='steelblue', markersize=8)
axes[0, 0].set_title('Total Grid Cells vs Cell Size')
axes[0, 0].set_xlabel('Grid Cell (m)'); axes[0, 0].set_ylabel('Total Cells')
axes[0, 0].set_yscale('log')

# Active cells vs grain
axes[0, 1].plot(exp3_df['grid_cell_m'], exp3_df['n_active_cells'], 's-', color='coral', markersize=8)
axes[0, 1].set_title('Active Cells vs Cell Size')
axes[0, 1].set_xlabel('Grid Cell (m)'); axes[0, 1].set_ylabel('Active Cells')

# Active ratio
active_ratio = exp3_df['n_active_cells'] / exp3_df['n_cells']
axes[0, 2].plot(exp3_df['grid_cell_m'], active_ratio * 100, '^-', color='green', markersize=8)
axes[0, 2].set_title('Cell Utilization Rate')
axes[0, 2].set_xlabel('Grid Cell (m)'); axes[0, 2].set_ylabel('Active Cells / Total Cells (%)')

# Compute time vs grain
axes[1, 0].plot(exp3_df['grid_cell_m'], exp3_df['compute_ms'], 'D-', color='purple', markersize=8)
axes[1, 0].set_title('Compute Time vs Cell Size')
axes[1, 0].set_xlabel('Grid Cell (m)'); axes[1, 0].set_ylabel('Compute Time (ms)')

# Bandwidth cells vs grain
axes[1, 1].plot(exp3_df['grid_cell_m'], exp3_df['bandwidth_cells'], 'p-', color='brown', markersize=8)
axes[1, 1].set_title('Kernel Radius (cells) vs Cell Size')
axes[1, 1].set_xlabel('Grid Cell (m)'); axes[1, 1].set_ylabel('Kernel Radius (cells)')

# Hotspot count vs grain
axes[1, 2].plot(exp3_df['grid_cell_m'], exp3_df['n_hotspots'], 'h-', color='teal', markersize=8)
axes[1, 2].set_title('Hotspot Count vs Cell Size')
axes[1, 2].set_xlabel('Grid Cell (m)'); axes[1, 2].set_ylabel('Hotspots')

plt.suptitle('Experiment 3: Grid Resolution Effects', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp3_grid_resolution.png', bbox_inches='tight')
plt.show()

In [ ]:
# Hotspot spatial distribution comparison
fig = plot_hotspot_comparison(
    {k: v for k, v in list(exp3_results.items())[:6]},
    exp1_lons, exp1_lats, n_cols=3, figsize=(18, 11)
)
fig.suptitle('Experiment 3: Grid Resolution - Hotspot Spatial Distribution', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp3_grid_hotspots.png', bbox_inches='tight')
plt.show()

---
## 7. Experiment 4: Kernel Function Comparison

**Research Question:** Which kernel function produces the most interpretable and stable hotspot results?

**Method:** Run STKDE with same parameters but different kernel functions (Gaussian, Epanechnikov, Uniform, Quartic, Triangular).

In [ ]:
KERNEL_NAMES = ['gaussian', 'epanechnikov', 'uniform', 'quartic', 'triangular']

exp4_results = {}
exp4_stats = []
exp4_intensities = {}  # Store full intensity grids for comparison

for kname in KERNEL_NAMES:
    config = STKDEConfig(
        spatial_bw_m=750, temporal_bw_h=24, grid_cell_m=500,
        top_k=15, min_support=5, time_window_h=24, kernel=kname
    )
    engine = STKDEEngine(config)
    hotspots, stats = engine.run(exp1_lons, exp1_lats, exp1_ts)
    stats['kernel'] = kname
    exp4_results[kname.capitalize()] = (hotspots, config)
    exp4_stats.append(stats)
    exp4_intensities[kname] = engine._intensity.copy()
    
    # Top 3 hotspot intensities
    top_intensities = [f'{h["intensity"]:.4f}' for h in hotspots[:3]]
    print(f'  {kname:>15}: {len(hotspots)} hotspots, intensities={top_intensities}, '
          f'max_intensity={stats["max_intensity"]:.1f}, compute={stats["compute_ms"]}ms')

print('\nExperiment 4 complete.')

In [ ]:
# Visual comparison of hotspots across kernels
fig = plot_hotspot_comparison(exp4_results, exp1_lons, exp1_lats, n_cols=3, figsize=(18, 11))
fig.suptitle('Experiment 4: Kernel Function Comparison', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp4_kernel_hotspots.png', bbox_inches='tight')
plt.show()

In [ ]:
# Intensity distribution comparison across kernels
fig, axes = plt.subplots(1, len(KERNEL_NAMES), figsize=(20, 4))

for ax, kname in zip(axes, KERNEL_NAMES):
    intensity = exp4_intensities[kname]
    nonzero = intensity[intensity > 0]
    ax.hist(nonzero, bins=50, color='steelblue', alpha=0.7, edgecolor='white')
    ax.axvline(x=np.percentile(nonzero, 95), color='red', linestyle='--', 
              label=f'P95={np.percentile(nonzero, 95):.1f}')
    ax.set_title(f'{kname.capitalize()}\n(nonzero cells: {len(nonzero):,})')
    ax.set_xlabel('Raw Intensity'); ax.set_ylabel('Frequency')
    ax.legend(fontsize=7)

plt.suptitle('Experiment 4: Intensity Distribution by Kernel Function', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp4_intensity_distributions.png', bbox_inches='tight')
plt.show()

In [ ]:
# Quantitative kernel comparison
exp4_df = pd.DataFrame(exp4_stats)

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Number of non-zero cells
axes[0, 0].bar(exp4_df['kernel'], exp4_df['n_active_cells'], color='steelblue')
axes[0, 0].set_title('Active Cells by Kernel')
axes[0, 0].set_ylabel('Active Cells')
axes[0, 0].tick_params(axis='x', rotation=45)

# Max intensity
axes[0, 1].bar(exp4_df['kernel'], exp4_df['max_intensity'], color='coral')
axes[0, 1].set_title('Max Intensity by Kernel')
axes[0, 1].set_ylabel('Max Intensity')
axes[0, 1].tick_params(axis='x', rotation=45)

# Compute time
axes[1, 0].bar(exp4_df['kernel'], exp4_df['compute_ms'], color='green')
axes[1, 0].set_title('Compute Time by Kernel')
axes[1, 0].set_ylabel('Time (ms)')
axes[1, 0].tick_params(axis='x', rotation=45)

# Hotspot count
axes[1, 1].bar(exp4_df['kernel'], exp4_df['n_hotspots'], color='purple')
axes[1, 1].set_title('Hotspot Count by Kernel')
axes[1, 1].set_ylabel('Hotspots')
axes[1, 1].tick_params(axis='x', rotation=45)

plt.suptitle('Experiment 4: Kernel Function Metrics', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp4_kernel_metrics.png', bbox_inches='tight')
plt.show()

---
## 8. Experiment 5: Combined Parameter Sweep (SBW x Grid)

**Research Question:** What is the joint effect of spatial bandwidth and grid resolution on hotspot quality?

**Method:** 2D parameter sweep over spatial bandwidth and grid cell size.

In [ ]:
# 2D sweep: Spatial BW x Grid Cell Size
SWEEP_SBW = [300, 500, 750, 1000, 1500, 2000, 3000]
SWEEP_GRID = [200, 400, 600, 800, 1000, 1500]

sweep_results = {}
sweep_matrix = {
    'n_hotspots': np.zeros((len(SWEEP_GRID), len(SWEEP_SBW))),
    'n_active_cells': np.zeros((len(SWEEP_GRID), len(SWEEP_SBW))),
    'max_intensity': np.zeros((len(SWEEP_GRID), len(SWEEP_SBW))),
    'compute_ms': np.zeros((len(SWEEP_GRID), len(SWEEP_SBW))),
    'bandwidth_cells': np.zeros((len(SWEEP_GRID), len(SWEEP_SBW))),
}

total = len(SWEEP_GRID) * len(SWEEP_SBW)
count = 0

for i, gcm in enumerate(SWEEP_GRID):
    for j, sbw in enumerate(SWEEP_SBW):
        # Skip invalid combos (kernel smaller than grid cell)
        if sbw < gcm * 0.5:
            for key in sweep_matrix:
                sweep_matrix[key][i, j] = np.nan
            continue
        
        config = STKDEConfig(
            spatial_bw_m=sbw, temporal_bw_h=24, grid_cell_m=gcm,
            top_k=15, min_support=5, time_window_h=24, kernel='gaussian'
        )
        engine = STKDEEngine(config)
        _, stats = engine.run(exp1_lons, exp1_lats, exp1_ts)
        
        sweep_matrix['n_hotspots'][i, j] = stats['n_hotspots']
        sweep_matrix['n_active_cells'][i, j] = stats['n_active_cells']
        sweep_matrix['max_intensity'][i, j] = stats['max_intensity']
        sweep_matrix['compute_ms'][i, j] = stats['compute_ms']
        sweep_matrix['bandwidth_cells'][i, j] = stats['bandwidth_cells']
        
        count += 1
        print(f'  [{count}/{total}] SBW={sbw}m, Grid={gcm}m: '
              f'{stats["n_hotspots"]} hotspots, {stats["compute_ms"]}ms')

print(f'\nExperiment 5 complete: {count} parameter combinations evaluated.')

In [ ]:
# Heatmap visualization of the 2D sweep
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

metrics = [
    ('n_hotspots', 'Number of Hotspots', 'YlOrRd'),
    ('n_active_cells', 'Active Cells', 'YlOrRd'),
    ('max_intensity', 'Max Intensity', 'viridis'),
    ('compute_ms', 'Compute Time (ms)', 'YlOrRd_r'),
    ('bandwidth_cells', 'Kernel Radius (cells)', 'YlOrRd'),
]

for ax, (metric, title, cmap) in zip(axes.flatten(), metrics):
    data = sweep_matrix[metric]
    masked = np.ma.masked_invalid(data)
    im = ax.pcolormesh(SWEEP_SBW, SWEEP_GRID, masked, cmap=cmap, shading='auto')
    plt.colorbar(im, ax=ax, label=title, shrink=0.8)
    ax.set_title(title)
    ax.set_xlabel('Spatial Bandwidth (m)')
    ax.set_ylabel('Grid Cell Size (m)')

# Hide unused subplot
axes[1, 2].set_visible(False)

plt.suptitle('Experiment 5: Parameter Interaction Heatmaps (SBW x Grid)', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp5_parameter_sweep.png', bbox_inches='tight')
plt.show()

---
## 9. Experiment 6: Hotspot Stability & Robustness

**Research Question:** How stable are STKDE hotspots under data perturbation and parameter perturbation?

**Method:** Bootstrap resampling of input data, measure hotspot centroid consistency.

In [ ]:
# Bootstrap stability analysis
N_BOOTSTRAP = 20
BOOTSTRAP_FRACTION = 0.8

base_config = STKDEConfig(
    spatial_bw_m=750, temporal_bw_h=24, grid_cell_m=500,
    top_k=10, min_support=5, time_window_h=24, kernel='gaussian'
)

n_total = len(exp1_lons)
bootstrap_hotspots = []

print(f'Running {N_BOOTSTRAP} bootstrap iterations (sampling {BOOTSTRAP_FRACTION*100:.0f}% each)...')
for i in range(N_BOOTSTRAP):
    idx = np.random.choice(n_total, int(n_total * BOOTSTRAP_FRACTION), replace=True)
    engine = STKDEEngine(base_config)
    hotspots, _ = engine.run(exp1_lons[idx], exp1_lats[idx], exp1_ts[idx])
    bootstrap_hotspots.append(hotspots)
    print(f'  Bootstrap {i+1}/{N_BOOTSTRAP}: {len(hotspots)} hotspots', end='\r')

print(f'\nBootstrap analysis complete.')

In [ ]:
# Analyze centroid stability
from collections import Counter

# Collect all hotspot centroids across bootstraps
all_centroids = []
for hotspots in bootstrap_hotspots:
    for h in hotspots:
        all_centroids.append((h['lon'], h['lat'], h['intensity']))

# Cluster centroids by proximity (simple greedy)
CLUSTER_RADIUS_M = 500  # meters
cluster_radius_deg = meters_to_lat_degrees(CLUSTER_RADIUS_M)

clusters = []
assigned = set()

all_centroids_sorted = sorted(enumerate(all_centroids), key=lambda x: x[1][2], reverse=True)

for idx, (lon, lat, intensity) in all_centroids_sorted:
    if idx in assigned:
        continue
    cluster = [(lon, lat, intensity)]
    assigned.add(idx)
    for j, (lon2, lat2, intensity2) in all_centroids_sorted:
        if j in assigned:
            continue
        dlat = abs(lat - lat2) * METERS_PER_LAT_DEGREE
        dlon = abs(lon - lon2) * METERS_PER_LAT_DEGREE * np.cos(np.radians((lat + lat2) / 2))
        dist = np.sqrt(dlat**2 + dlon**2)
        if dist < CLUSTER_RADIUS_M:
            cluster.append((lon2, lat2, intensity2))
            assigned.add(j)
    clusters.append(cluster)

# Sort clusters by recurrence (count) and mean intensity
clusters.sort(key=lambda c: len(c), reverse=True)

# Compute stability metrics
stability_data = []
for i, cluster in enumerate(clusters[:15]):
    lons = [c[0] for c in cluster]
    lats = [c[1] for c in cluster]
    intensities = [c[2] for c in cluster]
    
    stability_data.append({
        'cluster_id': i + 1,
        'count': len(cluster),
        'recurrence_rate': len(cluster) / N_BOOTSTRAP,
        'mean_lon': np.mean(lons),
        'mean_lat': np.mean(lats),
        'std_lon_m': np.std(lons) * METERS_PER_LAT_DEGREE * np.cos(np.radians(np.mean(lats))),
        'std_lat_m': np.std(lats) * METERS_PER_LAT_DEGREE,
        'mean_intensity': np.mean(intensities),
        'std_intensity': np.std(intensities),
    })

stability_df = pd.DataFrame(stability_data)
print('Top 10 Stable Hotspot Clusters:')
print(stability_df.head(10).to_string(index=False))

In [ ]:
# Plot bootstrap stability results
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Recurrence rate distribution
axes[0, 0].bar(stability_df['cluster_id'], stability_df['recurrence_rate'], color='steelblue')
axes[0, 0].axhline(y=0.5, color='red', linestyle='--', alpha=0.5, label='50% recurrence')
axes[0, 0].set_title('Hotspot Recurrence Rate Across Bootstraps')
axes[0, 0].set_xlabel('Cluster ID'); axes[0, 0].set_ylabel('Recurrence Rate')
axes[0, 0].legend()

# Spatial std vs recurrence
spatial_std = np.sqrt(stability_df['std_lon_m']**2 + stability_df['std_lat_m']**2)
axes[0, 1].scatter(stability_df['recurrence_rate'], spatial_std,
                  s=stability_df['count']*5, alpha=0.7, c='steelblue')
axes[0, 1].set_title('Spatial Precision vs Recurrence')
axes[0, 1].set_xlabel('Recurrence Rate'); axes[0, 1].set_ylabel('Spatial Std Dev (m)')

# Intensity stability
axes[1, 0].errorbar(stability_df['cluster_id'], stability_df['mean_intensity'],
                   yerr=stability_df['std_intensity'], fmt='o', capsize=4,
                   color='steelblue', markersize=6)
axes[1, 0].set_title('Intensity Stability (Mean +/- Std)')
axes[1, 0].set_xlabel('Cluster ID'); axes[1, 0].set_ylabel('Intensity')

# Hotspot cluster map
n_sample_pts = min(20000, len(exp1_lons))
idx_pts = np.random.choice(len(exp1_lons), n_sample_pts, replace=False)
axes[1, 1].scatter(exp1_lons[idx_pts], exp1_lats[idx_pts], s=1, alpha=0.15, c='#888888')

colors = plt.cm.viridis(np.linspace(0, 1, len(stability_df.head(10))))
for i, row in stability_df.head(10).iterrows():
    axes[1, 1].scatter(row['mean_lon'], row['mean_lat'],
                      s=row['count'] * 8, alpha=0.7, color=colors[i],
                      edgecolors='black', linewidth=0.5)
    axes[1, 1].annotate(f"{int(row['cluster_id'])}", (row['mean_lon'], row['mean_lat']),
                       fontsize=7, ha='center', va='center', fontweight='bold')

axes[1, 1].set_title('Stable Hotspot Clusters')
axes[1, 1].set_xlabel('Longitude'); axes[1, 1].set_ylabel('Latitude')

plt.suptitle('Experiment 6: Hotspot Stability Analysis', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp6_stability.png', bbox_inches='tight')
plt.show()

# Summary statistics
print(f'\nStability Summary:')
print(f'  Unique hotspot clusters: {len(clusters)}')
print(f'  Clusters with >50% recurrence: {(stability_df["recurrence_rate"] > 0.5).sum()}')
print(f'  Mean recurrence rate (top 10): {stability_df["recurrence_rate"].head(10).mean():.3f}')
print(f'  Mean spatial std dev: {spatial_std.mean():.1f}m')

---
## 10. Experiment 7: minSupport Sensitivity

**Research Question:** How does the minimum support threshold affect hotspot detection and false positives?

**Method:** Sweep minSupport from 1 to 100, track hotspot count and intensity distribution.

In [ ]:
MIN_SUPPORT_VALUES = [1, 3, 5, 10, 20, 50, 100]

exp7_results = {}
exp7_stats = []

for ms in MIN_SUPPORT_VALUES:
    config = STKDEConfig(
        spatial_bw_m=750, temporal_bw_h=24, grid_cell_m=500,
        top_k=50, min_support=ms, time_window_h=24, kernel='gaussian'
    )
    engine = STKDEEngine(config)
    hotspots, stats = engine.run(exp1_lons, exp1_lats, exp1_ts)
    stats['min_support'] = ms
    exp7_results[f'MinSup={ms}'] = (hotspots, config)
    exp7_stats.append(stats)
    print(f'  MinSupport={ms}: {stats["n_hotspot_candidates"]} candidates -> {len(hotspots)} hotspots (filtered)')

print('\nExperiment 7 complete.')

In [ ]:
exp7_df = pd.DataFrame(exp7_stats)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(exp7_df['min_support'], exp7_df['n_hotspot_candidates'], 'o-', color='steelblue', 
            markersize=8, label='Candidates')
axes[0].plot(exp7_df['min_support'], exp7_df['n_hotspots'], 's-', color='coral', 
            markersize=8, label='Top-K Hotspots')
axes[0].set_title('Hotspot Candidates vs minSupport')
axes[0].set_xlabel('minSupport'); axes[0].set_ylabel('Count')
axes[0].legend()
axes[0].set_xscale('log')

axes[1].plot(exp7_df['min_support'], exp7_df['max_intensity'], '^-', color='green', markersize=8)
axes[1].set_title('Max Intensity vs minSupport')
axes[1].set_xlabel('minSupport'); axes[1].set_ylabel('Max Intensity')
axes[1].set_xscale('log')

plt.suptitle('Experiment 7: minSupport Sensitivity', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'exp7_minsupport.png', bbox_inches='tight')
plt.show()

---
## 11. Summary & Thesis-Ready Figures

In [ ]:
# Compile all experiment results into a summary DataFrame
all_results = []

for exp_name, exp_stats in [
    ('Exp1_SpatialBW', exp1_stats),
    ('Exp2_TemporalBW', exp2_stats),
    ('Exp3_GridResolution', exp3_stats),
    ('Exp4_Kernel', exp4_stats),
    ('Exp7_MinSupport', exp7_stats),
]:
    for s in exp_stats:
        s['experiment'] = exp_name
        all_results.append(s)

summary_df = pd.DataFrame(all_results)
summary_df.to_csv(OUTPUT_DIR / 'all_experiment_results.csv', index=False)

print('=== STKDE Experiment Summary ===')
print(f'\nTotal experiments run: {len(all_results)}')
print(f'Results saved to: {OUTPUT_DIR / "all_experiment_results.csv"}')
print()

print('Key Findings:')
print(f'  1. Spatial bandwidth: Increasing SBW from {SPATIAL_BW_VALUES[0]}m to {SPATIAL_BW_VALUES[-1]}m '
      f'changes hotspot count from '
      f'{exp1_stats[0]["n_hotspots"]} to {exp1_stats[-1]["n_hotspots"]}')
print(f'  2. Grid resolution: Finest grid ({GRID_CELL_VALUES[0]}m) creates '
      f'{exp3_stats[0]["n_cells"]:,} cells, coarsest ({GRID_CELL_VALUES[-1]}m) creates '
      f'{exp3_stats[-1]["n_cells"]:,} cells')
print(f'  3. Kernel choice: Gaussian produces {exp4_stats[0]["n_active_cells"]:,} active cells, '
      f'vs Epanechnikov {exp4_stats[1]["n_active_cells"]:,}')
print(f'  4. Hotspot stability: Mean recurrence rate = {stability_df["recurrence_rate"].head(10).mean():.3f}')

print(f'\nFigures saved to: {OUTPUT_DIR.resolve()}')
print('Notebook execution complete.')

In [ ]:
# Final thesis-quality summary figure
fig = plt.figure(figsize=(18, 12))
gs = fig.add_gridspec(3, 3, hspace=0.35, wspace=0.35)

# (1) Spatial BW -> Hotspots
ax1 = fig.add_subplot(gs[0, 0])
exp1_df_sum = pd.DataFrame(exp1_stats)
ax1.plot(exp1_df_sum['spatial_bw_m'], exp1_df_sum['n_hotspots'], 'o-', color='#2196F3', lw=2, ms=8)
ax1.fill_between(exp1_df_sum['spatial_bw_m'], 0, exp1_df_sum['n_hotspots'], alpha=0.15, color='#2196F3')
ax1.set_xlabel('Spatial Bandwidth (m)'); ax1.set_ylabel('Hotspots')
ax1.set_title('(a) Spatial BW vs Hotspot Count', fontweight='bold')

# (2) Grid Resolution -> Cells
ax2 = fig.add_subplot(gs[0, 1])
exp3_df_sum = pd.DataFrame(exp3_stats)
ax2.plot(exp3_df_sum['grid_cell_m'], exp3_df_sum['n_active_cells'], 's-', color='#FF5722', lw=2, ms=8)
ax2.fill_between(exp3_df_sum['grid_cell_m'], 0, exp3_df_sum['n_active_cells'], alpha=0.15, color='#FF5722')
ax2.set_xlabel('Grid Cell Size (m)'); ax2.set_ylabel('Active Cells')
ax2.set_title('(b) Grid Resolution vs Active Cells', fontweight='bold')

# (3) Kernel Comparison
ax3 = fig.add_subplot(gs[0, 2])
exp4_df_sum = pd.DataFrame(exp4_stats)
colors_kernel = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']
ax3.bar(range(len(exp4_df_sum)), exp4_df_sum['n_active_cells'], color=colors_kernel, edgecolor='white')
ax3.set_xticks(range(len(exp4_df_sum)))
ax3.set_xticklabels(exp4_df_sum['kernel'], rotation=30, ha='right')
ax3.set_ylabel('Active Cells')
ax3.set_title('(c) Kernel Function Impact', fontweight='bold')

# (4) Parameter Sweep Heatmap
ax4 = fig.add_subplot(gs[1, :])
im = ax4.pcolormesh(SWEEP_SBW, SWEEP_GRID, sweep_matrix['n_hotspots'], 
                   cmap='YlOrRd', shading='auto')
plt.colorbar(im, ax=ax4, label='Hotspot Count', shrink=0.8)
ax4.set_xlabel('Spatial Bandwidth (m)'); ax4.set_ylabel('Grid Cell Size (m)')
ax4.set_title('(d) Parameter Interaction: SBW x Grid Resolution', fontweight='bold')

# (5) Stability - Recurrence
ax5 = fig.add_subplot(gs[2, :2])
ax5.bar(stability_df['cluster_id'].head(12), stability_df['recurrence_rate'].head(12), 
       color='#607D8B', edgecolor='white')
ax5.axhline(y=0.5, color='#F44336', linestyle='--', lw=2, alpha=0.7, label='50% threshold')
ax5.set_xlabel('Hotspot Cluster ID'); ax5.set_ylabel('Recurrence Rate')
ax5.set_title('(e) Hotspot Stability (Bootstrap Recurrence)', fontweight='bold')
ax5.legend()
ax5.set_ylim(0, 1)

# (6) Runtime comparison
ax6 = fig.add_subplot(gs[2, 2])
ax6.plot(exp3_df_sum['grid_cell_m'], exp3_df_sum['compute_ms'], 'D-', color='#795548', lw=2, ms=8)
ax6.set_xlabel('Grid Cell Size (m)'); ax6.set_ylabel('Compute Time (ms)')
ax6.set_title('(f) Computational Cost', fontweight='bold')

plt.suptitle('STKDE Parameter Sensitivity Analysis — Thesis Summary', 
             fontsize=16, fontweight='bold', y=1.01)
plt.savefig(OUTPUT_DIR / 'thesis_summary_figure.png', dpi=300, bbox_inches='tight',
           facecolor='white', edgecolor='none')
plt.show()
print('Thesis summary figure saved.')